In [5]:
import re
import json
from collections import defaultdict
import os
import math

## **Preprocessing & Indexing**

In [6]:
# path corpus dan stopwords dari tugas2
CORPUS_PATH = "../../data/tugas2/corpus.txt"
STOPWORD_PATH = "../../data/tugas2/id.stopwords.02.01.2016.txt"

In [7]:
# load stopwords
def load_stopwords(file_path):
    stopwords = set()
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            stopwords.add(line.strip())
    return stopwords

In [8]:
# preprocessing
def preprocess(text, stopwords):
    # case folding
    text = text.lower()
    
    # remove non alphabet
    text = re.sub(r'[^a-z\s]', '', text)
    
    # tokenizing
    tokens = text.split()
    
    # remove stopwords
    tokens = [word for word in tokens if word not in stopwords]
    
    return tokens

In [9]:
def load_corpus(file_path, stopwords):
    documents = {}

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    docs = re.findall(r'<DOC>(.*?)</DOC>', content, re.DOTALL)

    for i, doc in enumerate(docs, start=1):
        title_match = re.search(r'<TITLE>(.*?)</TITLE>', doc, re.DOTALL)
        text_match = re.search(r'<TEXT>(.*?)</TEXT>', doc, re.DOTALL)

        title = title_match.group(1).strip() if title_match else ""
        text = text_match.group(1).strip() if text_match else ""

        full_text = title + " " + text
        tokens = preprocess(full_text, stopwords)

        documents[i] = {
            "title": title,
            "content": text,
            "tokens": tokens
        }

    return documents

In [10]:
# inverted list
def build_inverted_index(documents):
    inverted_index = defaultdict(dict)

    for doc_id, doc in documents.items():
        tf = defaultdict(int)

        for token in doc["tokens"]:
            tf[token] += 1

        for term, freq in tf.items():
            inverted_index[term][doc_id] = freq

    return dict(inverted_index)

In [11]:
# save json
def save_json(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)


In [12]:
# main
if __name__ == "__main__":
    print("Loading stopwords...")
    stopwords = load_stopwords(STOPWORD_PATH)

    print("Processing corpus...")
    documents = load_corpus(CORPUS_PATH, stopwords)

    print("Building inverted index...")
    inverted_index = build_inverted_index(documents)

    save_json(documents, "documents.json")
    save_json(inverted_index, "inverted_index.json")

    print("\nDONE")
    print(f"Total documents: {len(documents)}")
    print(f"Total terms: {len(inverted_index)}")

Loading stopwords...
Processing corpus...
Building inverted index...

DONE
Total documents: 479
Total terms: 14157


## **Pembobotan Kata (TF-IDF) & Representasi Vektor**

Bagian ini mengimplementasikan:
1. Perhitungan **TF (Term Frequency)** dari inverted index
2. Perhitungan **IDF (Inverse Document Frequency)** dari inverted index
3. Perhitungan **TF-IDF** untuk setiap term di setiap dokumen
4. **Representasi vektor dokumen** berbasis TF-IDF
5. Penyimpanan hasil ke file JSON untuk digunakan bagian retrieval

**Load Data**

In [13]:
with open('inverted_index.json', 'r', encoding='utf-8') as f:
    inverted_index = json.load(f)

with open('documents.json', 'r', encoding='utf-8') as f:
    documents = json.load(f)

print(f'Jumlah term dalam inverted index : {len(inverted_index):,}')
print(f'Jumlah dokumen                   : {len(documents):,}')

Jumlah term dalam inverted index : 14,157
Jumlah dokumen                   : 479


**Hitung TF-IDF**

TF

In [14]:
def compute_tf(inverted_index, documents):
    doc_length = {}
    for doc_id, doc_data in documents.items():
        doc_length[doc_id] = len(doc_data['tokens'])
    
    tf = defaultdict(dict)
    
    for term, postings in inverted_index.items():
        for doc_id, raw_freq in postings.items():
            doc_len = doc_length.get(doc_id, 1)
            tf[doc_id][term] = raw_freq / doc_len
    
    return dict(tf)

tf = compute_tf(inverted_index, documents)

In [15]:
# cek
sample_doc = list(tf.keys())[0]
sample_terms = list(tf[sample_doc].items())[:5]
print(f'Contoh TF untuk dokumen "{sample_doc}":')
for term, val in sample_terms:
    print(f'  TF("{term}", doc {sample_doc}) = {val:.6f}')

Contoh TF untuk dokumen "1":
  TF("cepat", doc 1) = 0.015094
  TF("turunkan", doc 1) = 0.003774
  TF("kadar", doc 1) = 0.041509
  TF("gula", doc 1) = 0.067925
  TF("darah", doc 1) = 0.056604


IDF

In [16]:
def compute_idf(inverted_index, N):
    idf = {}
    for term, postings in inverted_index.items():
        df = len(postings)
        idf[term] = math.log(N / df) + 1
    return idf

N = len(documents)
idf = compute_idf(inverted_index, N)

sorted_idf = sorted(idf.items(), key=lambda x: x[1])

In [17]:
# verifikasi
print(f'Total term yang dihitung IDF-nya: {len(idf):,}')
print()
print('10 Term dengan IDF TERENDAH (paling umum):')
for term, val in sorted_idf[:10]:
    df = len(inverted_index[term])
    print(f'  IDF("{term}") = {val:.4f}  [muncul di {df} dokumen]')

print()
print('10 Term dengan IDF TERTINGGI (paling jarang/spesifik):')
for term, val in sorted_idf[-10:]:
    df = len(inverted_index[term])
    print(f'  IDF("{term}") = {val:.4f}  [muncul di {df} dokumen]')

Total term yang dihitung IDF-nya: 14,157

10 Term dengan IDF TERENDAH (paling umum):
  IDF("to") = 1.6110  [muncul di 260 dokumen]
  IDF("scroll") = 1.6149  [muncul di 259 dokumen]
  IDF("with") = 1.6149  [muncul di 259 dokumen]
  IDF("continue") = 1.6187  [muncul di 258 dokumen]
  IDF("content") = 1.6187  [muncul di 258 dokumen]
  IDF("indonesia") = 1.9732  [muncul di 181 dokumen]
  IDF("advertisement") = 2.0537  [muncul di 167 dokumen]
  IDF("memiliki") = 2.0537  [muncul di 167 dokumen]
  IDF("salah") = 2.0718  [muncul di 164 dokumen]
  IDF("kompascom") = 2.3354  [muncul di 126 dokumen]

10 Term dengan IDF TERTINGGI (paling jarang/spesifik):
  IDF("tebesaya") = 7.1717  [muncul di 1 dokumen]
  IDF("batuan") = 7.1717  [muncul di 1 dokumen]
  IDF("tersohor") = 7.1717  [muncul di 1 dokumen]
  IDF("bendi") = 7.1717  [muncul di 1 dokumen]
  IDF("djudjul") = 7.1717  [muncul di 1 dokumen]
  IDF("ketut") = 7.1717  [muncul di 1 dokumen]
  IDF("lukisnya") = 7.1717  [muncul di 1 dokumen]
  IDF("

TF-IDF

In [18]:
def compute_tfidf(tf, idf):
    tfidf = {}
    for doc_id, term_tf in tf.items():
        tfidf[doc_id] = {}
        for term, tf_val in term_tf.items():
            idf_val = idf.get(term, 0)
            tfidf[doc_id][term] = tf_val * idf_val
    return tfidf

tfidf = compute_tfidf(tf, idf)

In [19]:
# cek
sample_doc_id = '1'
doc_tfidf_sorted = sorted(tfidf[sample_doc_id].items(), key=lambda x: x[1], reverse=True)

print(f'Dokumen ID {sample_doc_id}: "{documents[sample_doc_id]["title"]}"')
print()
print('10 term paling penting (TF-IDF tertinggi):')
for term, val in doc_tfidf_sorted[:10]:
    print(f'  TFIDF("{term}", doc {sample_doc_id}) = {val:.6f}')

Dokumen ID 1: "4 Cara Cepat Turunkan Kadar Gula Darah Saat Tiba-tiba Naik"

10 term paling penting (TF-IDF tertinggi):
  TFIDF("gula", doc 1) = 0.290807
  TFIDF("darah", doc 1) = 0.217330
  TFIDF("kadar", doc 1) = 0.188147
  TFIDF("insulin", doc 1) = 0.160421
  TFIDF("glukosa", doc 1) = 0.101508
  TFIDF("diabetes", doc 1) = 0.096080
  TFIDF("hong", doc 1) = 0.096080
  TFIDF("obat", doc 1) = 0.078792
  TFIDF("menurunkan", doc 1) = 0.065487
  TFIDF("komplikasi", doc 1) = 0.062969


**Representasi Vektor**

In [20]:
def build_doc_vectors(tfidf):
    doc_vectors = {}
    for doc_id, term_weights in tfidf.items():
        doc_vectors[doc_id] = {term: weight 
                                for term, weight in term_weights.items() 
                                if weight > 0}
    return doc_vectors

doc_vectors = build_doc_vectors(tfidf)

In [21]:
# cek
vec_lengths = [len(v) for v in doc_vectors.values()]
print(f'Total dokumen yang direpresentasikan : {len(doc_vectors):,}')
print(f'Rata-rata dimensi vektor per dokumen : {sum(vec_lengths)/len(vec_lengths):.1f} term')
print(f'Dimensi vektor min                  : {min(vec_lengths)}')
print(f'Dimensi vektor max                  : {max(vec_lengths)}')

Total dokumen yang direpresentasikan : 479
Rata-rata dimensi vektor per dokumen : 121.4 term
Dimensi vektor min                  : 1
Dimensi vektor max                  : 382


**Simpan Hasil ke JSON**

In [22]:
with open('tfidf_weights.json', 'w', encoding='utf-8') as f:
    json.dump(tfidf, f, ensure_ascii=False)
print('✅ tfidf_weights.json tersimpan')

with open('idf_values.json', 'w', encoding='utf-8') as f:
    json.dump(idf, f, ensure_ascii=False)
print('✅ idf_values.json tersimpan')

with open('doc_vectors.json', 'w', encoding='utf-8') as f:
    json.dump(doc_vectors, f, ensure_ascii=False)
print('✅ doc_vectors.json tersimpan')

✅ tfidf_weights.json tersimpan
✅ idf_values.json tersimpan
✅ doc_vectors.json tersimpan
